# 06 — Model Explainability with Grad-CAM

In medical AI, **explainability is not optional** — clinicians must understand  
why the model made a decision to trust it in practice.

**Grad-CAM** (Gradient-weighted Class Activation Mapping) highlights which  
regions of the X-ray most influenced the model's prediction.

### What we expect to see:
- **PNEUMONIA**: Activation in lung consolidation areas (typically basal/lobar regions)
- **NORMAL**: Diffuse/minimal activation — no focal abnormality

### Also covered:
- Misclassification analysis (what does the model get wrong?)
- Expert system finding overlay
- Uncertainty analysis via Test-Time Augmentation (TTA)

In [ ]:
import sys
sys.path.insert(0, '..')

import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import cv2

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

from src.models import build_model, ChestExpertSystem
from src.data.dataset import get_transforms, get_tta_transforms, IDX_TO_CLASS

plt.rcParams['figure.dpi'] = 120
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_ROOT = Path('../chest_xray')

# Load model
model = build_model('densenet121', num_classes=2, pretrained=False, device=DEVICE)
ckpt = torch.load('../models_saved/best_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('Model loaded')

## 1. Grad-CAM Gallery

In [ ]:
# Target the last DenseNet denseblock for Grad-CAM
target_layer = model.features.denseblock4
cam = GradCAMPlusPlus(model=model, target_layers=[target_layer])

transform = get_transforms('test', 224)

def get_gradcam(img_path, label_str):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = transform(image=img)['image'].unsqueeze(0).to(DEVICE)
    
    grayscale_cam = cam(input_tensor=tensor)[0]
    img_resized = cv2.resize(img, (224, 224)).astype(np.float32) / 255.0
    overlay = show_cam_on_image(img_resized, grayscale_cam, use_rgb=True)
    
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    pred_label = IDX_TO_CLASS[probs.argmax().item()]
    prob_pneumonia = probs[1].item()
    
    return img_resized, grayscale_cam, overlay, pred_label, prob_pneumonia

random.seed(42)
samples = [
    (random.choice(list((DATA_ROOT / 'test' / 'NORMAL').glob('*.jpeg'))), 'NORMAL'),
    (random.choice(list((DATA_ROOT / 'test' / 'NORMAL').glob('*.jpeg'))), 'NORMAL'),
    (random.choice(list((DATA_ROOT / 'test' / 'PNEUMONIA').glob('*bacteria*'))), 'PNEUMONIA(Bacterial)'),
    (random.choice(list((DATA_ROOT / 'test' / 'PNEUMONIA').glob('*bacteria*'))), 'PNEUMONIA(Bacterial)'),
    (random.choice(list((DATA_ROOT / 'test' / 'PNEUMONIA').glob('*virus*'))), 'PNEUMONIA(Viral)'),
    (random.choice(list((DATA_ROOT / 'test' / 'PNEUMONIA').glob('*virus*'))), 'PNEUMONIA(Viral)'),
]

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(len(samples), 3, figure=fig, wspace=0.1, hspace=0.5)

for row, (path, true_label) in enumerate(samples):
    orig, cam_map, overlay, pred_label, prob = get_gradcam(path, true_label)
    correct = (true_label.startswith('PNEUMONIA')) == (pred_label == 'PNEUMONIA')
    color = 'green' if correct else 'red'
    
    ax0 = fig.add_subplot(gs[row, 0])
    ax0.imshow(orig, cmap='gray'); ax0.axis('off')
    ax0.set_title(f'True: {true_label}', fontsize=8, color='navy', fontweight='bold')
    
    ax1 = fig.add_subplot(gs[row, 1])
    ax1.imshow(cam_map, cmap='jet'); ax1.axis('off')
    ax1.set_title('Grad-CAM++', fontsize=8)
    
    ax2 = fig.add_subplot(gs[row, 2])
    ax2.imshow(overlay); ax2.axis('off')
    ax2.set_title(f'Pred: {pred_label} ({prob:.2%})', fontsize=8, color=color, fontweight='bold')

plt.suptitle('Grad-CAM++ Explainability — DenseNet-121 Predictions\n(Heatmap shows regions driving the classification decision)',
             fontsize=13, fontweight='bold')
plt.savefig('../reports/gradcam_gallery.png', bbox_inches='tight', dpi=150)
plt.show()

## 2. Test-Time Augmentation (TTA) — Uncertainty Estimation

In [ ]:
tta_transforms = get_tta_transforms(224)

def predict_with_tta(img_path):
    """Run TTA and return mean prediction + uncertainty."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    tta_probs = []
    with torch.no_grad():
        for t in tta_transforms:
            tensor = t(image=img)['image'].unsqueeze(0).to(DEVICE)
            prob = torch.softmax(model(tensor), dim=1)[0, 1].item()
            tta_probs.append(prob)
    
    return {
        'mean_prob':  np.mean(tta_probs),
        'std_prob':   np.std(tta_probs),
        'tta_probs':  tta_probs,
        'prediction': 'PNEUMONIA' if np.mean(tta_probs) >= 0.5 else 'NORMAL',
    }

random.seed(10)
test_cases = [
    (random.choice(list((DATA_ROOT / 'test' / 'NORMAL').glob('*.jpeg'))), 'NORMAL'),
    (random.choice(list((DATA_ROOT / 'test' / 'PNEUMONIA').glob('*.jpeg'))), 'PNEUMONIA'),
]

for path, true_label in test_cases:
    tta_result = predict_with_tta(path)
    print(f'True: {true_label:10s} | Pred: {tta_result["prediction"]:10s} | '
          f'Mean prob: {tta_result["mean_prob"]:.3f} ± {tta_result["std_prob"]:.3f} '
          f'| TTA: {[f"{p:.2f}" for p in tta_result["tta_probs"]]}')

## 3. Expert System + Grad-CAM Side-by-Side

In [ ]:
expert = ChestExpertSystem()

random.seed(55)
pneumonia_path = random.choice(list((DATA_ROOT / 'test' / 'PNEUMONIA').glob('*bacteria*')))
img = cv2.imread(str(pneumonia_path))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Expert analysis
findings = expert.analyze(img)
gray = expert.preprocess(img)
roi, mask = expert.extract_lung_roi(gray)

# Neural Grad-CAM
tensor = transform(image=img)['image'].unsqueeze(0).to(DEVICE)
cam_map = cam(input_tensor=tensor)[0]
img_224 = cv2.resize(img, (224, 224)).astype(np.float32) / 255.0
overlay = show_cam_on_image(img_224, cam_map, use_rgb=True)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img_224, cmap='gray'); axes[0].set_title('Original X-Ray', fontweight='bold'); axes[0].axis('off')
axes[1].imshow(cv2.resize(mask.astype(np.uint8) * 255, (224, 224)), cmap='Blues')
axes[1].set_title(f'Expert: Lung Mask\nScore={findings.final_score:.3f}', fontweight='bold'); axes[1].axis('off')
axes[2].imshow(cam_map, cmap='jet'); axes[2].set_title('NN: Grad-CAM++ Attention', fontweight='bold'); axes[2].axis('off')
axes[3].imshow(overlay); axes[3].set_title('Combined Overlay', fontweight='bold'); axes[3].axis('off')

fig.text(0.5, -0.02,
    f'Expert Findings — Opacity: {findings.opacity_score:.3f} | Texture: {findings.texture_score:.3f} | '
    f'Density: {findings.density_score:.3f} | Consolidation: {findings.consolidation_score:.3f}',
    ha='center', fontsize=10, color='darkred', fontweight='bold'
)
plt.suptitle('Hybrid Explainability: Expert Rules + Grad-CAM (PNEUMONIA)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/hybrid_explainability.png', bbox_inches='tight', dpi=150)
plt.show()